# Дообучение и оценка Llama 2

#### Загрузка предобученной модели

In [1]:
!pip install llama-stack

   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.6 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.6 MB 1.8 MB/s eta 0:00:02
   -------------------- ------------------- 1.8/3.6 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 3.6/3.6 MB 4.8 MB/s  0:00:01
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 16.4 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.4 MB 2.3 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.4 MB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.7 MB/s  0:00:00

   ----- ----------------------------------  2/14 [python-dotenv]
   -------- -------------------------------  3/14 [pyasn1]
   ----------- ----------------------------  4/14 [pyaml]
   ----------

In [4]:
!llama model list --show-all

+-----------------------------------------------------------------------------+
| Model Descriptor(ID)        | Hugging Face Repo            | Context Length |
|-----------------------------+------------------------------+----------------|
| Llama-2-7b                  | meta-llama/Llama-2-7b        | 4K             |
|-----------------------------+------------------------------+----------------|
| Llama-2-13b                 | meta-llama/Llama-2-13b       | 4K             |
|-----------------------------+------------------------------+----------------|
| Llama-2-70b                 | meta-llama/Llama-2-70b       | 4K             |
|-----------------------------+------------------------------+----------------|
| Llama-2-7b-chat             | meta-llama/Llama-2-7b-chat   | 4K             |
|-----------------------------+------------------------------+----------------|
| Llama-2-13b-chat            | meta-llama/Llama-2-13b-chat  | 4K             |
|-----------------------------+---------

C:\Users\vacil\kurs\finbert_env\lib\site-packages\pydantic\_internal\_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'default' attribute with value 'sqlite' was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


In [ ]:
!llama model download --source meta --model-id  Llama-2-7b-chat

In [1]:
import datasets
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)

from datasets import Dataset
import json
import re

LOCAL_MODEL_PATH = "./Llama-2-7b-chat-hb"


tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_PATH,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

print("Модель успешно загружена с локального пути")

E:\alena\fin_proj\fin_proj\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\alena\fin_proj\fin_proj\lib\site-packages\transformers\utils\hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you 

Модель успешно загружена с локального пути


In [2]:
import os

LOCAL_MODEL_PATH = "./Llama-2-7b-chat-hb" 
OUTPUT_DIR = "./llama2-ner-finetuned" 
TRAIN_FILE = "./data/train_mod.txt"
VAL_FILE = "./data/val_mod.txt"
TEST_FILE = "./data/test_mod.txt" 

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Модель: {LOCAL_MODEL_PATH}")
print(f"Обучающие файлы: {TRAIN_FILE}, {VAL_FILE}, {TEST_FILE}")

Модель: ./Llama-2-7b-chat-hb
Обучающие файлы: ./data/train_mod.txt, ./data/val_mod.txt, ./data/test_mod.txt


#### Конвертация датасета BIO в инструкции

In [4]:
def convert_bio_to_instructions(bio_file_path, output_json_path):
    label_mapping = {
        'PER': 'PERSON',
        'PERSON': 'PERSON',
        'ORG': 'ORGANIZATION', 
        'ORGANIZATION': 'ORGANIZATION',
        'LOC': 'LOCATION',
        'LOCATION': 'LOCATION',
        'MISC': 'MISC'
    }
    
    with open(bio_file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    

    sentences = []
    current_tokens = []
    current_labels = []
    
    for line in lines:
        line = line.strip()
        if not line: 
            if current_tokens:
                sentences.append((current_tokens, current_labels))
                current_tokens, current_labels = [], []
        else:
            parts = line.split()
            if len(parts) == 2:
                token, label = parts
                current_tokens.append(token)
                current_labels.append(label)
    
    if current_tokens:
        sentences.append((current_tokens, current_labels))
    
    instruction_data = []
    
    for tokens, labels in sentences:
        text = " ".join(tokens)
        

        entities = {"PERSON": [], "ORGANIZATION": [], "LOCATION": []}
        current_entity = []
        current_type = None
        
        for token, label in zip(tokens, labels):
            if label.startswith("B-"):
                if current_entity and current_type:
                    mapped_type = label_mapping.get(current_type, current_type)
                    if mapped_type in entities:
                        entities[mapped_type].append(" ".join(current_entity))
                
                raw_type = label[2:] 
                current_type = raw_type
                current_entity = [token]
                
            elif label.startswith("I-"):
                raw_type = label[2:]
                if current_entity and current_type == raw_type:
                    current_entity.append(token)
                else:
                    if current_entity and current_type:
                        mapped_type = label_mapping.get(current_type, current_type)
                        if mapped_type in entities:
                            entities[mapped_type].append(" ".join(current_entity))
                    current_entity = []
                    current_type = None
                    
            else:  # label == "O"
                if current_entity and current_type:
                    mapped_type = label_mapping.get(current_type, current_type)
                    if mapped_type in entities:
                        entities[mapped_type].append(" ".join(current_entity))
                current_entity = []
                current_type = None
        
        if current_entity and current_type:
            mapped_type = label_mapping.get(current_type, current_type)
            if mapped_type in entities:
                entities[mapped_type].append(" ".join(current_entity))
        
        def format_entity_list(entity_list):
            return ', '.join(entity_list) if entity_list else 'none'
        
        output = f"[Person]: {format_entity_list(entities['PERSON'])}\n"
        output += f"[Organization]: {format_entity_list(entities['ORGANIZATION'])}\n"
        output += f"[Location]: {format_entity_list(entities['LOCATION'])}"
        
        instruction = {
            "instruction": "You are an expert in Named Entity Recognition in finance. Please identify the Person, Organization, and Location Entity from the given text.",
            "input": text,
            "output": output
        }
        instruction_data.append(instruction)
    
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(instruction_data, f, indent=2, ensure_ascii=False)
    
    print(f"Конвертировано {len(instruction_data)} примеров в {output_json_path}")
    return instruction_data

In [5]:
print("Конвертация датасетов...")

train_data = convert_bio_to_instructions(TRAIN_FILE, "train_instructions.json")
val_data = convert_bio_to_instructions(VAL_FILE, "val_instructions.json")
test_data = convert_bio_to_instructions(TEST_FILE, "test_instructions.json")

if len(train_data) > 0:
    print("\nПример конвертированной инструкции:")
    print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

Конвертация датасетов...
Конвертировано 964 примеров в train_instructions.json
Конвертировано 200 примеров в val_instructions.json
Конвертировано 303 примеров в test_instructions.json

Пример конвертированной инструкции:
{
  "instruction": "You are an expert in Named Entity Recognition in finance. Please identify the Person, Organization, and Location Entity from the given text.",
  "input": "Section 10 . 1 Notices .",
  "output": "[Person]: none\n[Organization]: none\n[Location]: none"
}


#### Подготовка к обучению

In [6]:
def format_llama2_chat(instruction, input_text, output_text=None):
    system_prompt = f"<<SYS>>\n{instruction}\n<</SYS>>\n\n"
    
    if output_text:
        return f"<s>[INST] {system_prompt}{input_text} [/INST] {output_text} </s>"
    else:
        return f"<s>[INST] {system_prompt}{input_text} [/INST]"

test_format = format_llama2_chat(
    "Test instruction",
    "Test input",
    "Test output"
)

In [7]:
def prepare_dataset(instruction_data, tokenizer, max_length=512):
    texts = []
    for item in instruction_data:
        formatted_text = format_llama2_chat(
            item["instruction"],
            item["input"],
            item["output"]
        )
        texts.append(formatted_text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors=None
    )
    
    dataset_dict = {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": tokenized["input_ids"].copy()
    }
    
    return Dataset.from_dict(dataset_dict)

In [8]:
def prepare_dataset(instruction_data, tokenizer, max_length=512):
    texts = []
    for item in instruction_data:
        formatted_text = format_llama2_chat(
            item["instruction"],
            item["input"],
            item["output"]
        )
        texts.append(formatted_text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors=None
    )
    
    dataset_dict = {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": tokenized["input_ids"].copy() 
    }
    
    return Dataset.from_dict(dataset_dict)

In [9]:
def load_model_and_tokenizer():
    print("Загрузка модели и токенизатора...")
    
    tokenizer = AutoTokenizer.from_pretrained(
        LOCAL_MODEL_PATH,
        local_files_only=True
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    # 4-bit конфигурация
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_PATH,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        local_files_only=True
    )
    
    # подготовка для обучения с LoRA
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    
    print("Модель загружена")
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()

Загрузка модели и токенизатора...


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.11s/it]


Модель загружена


In [10]:
def setup_lora(model):
    lora_config = LoraConfig(
        r=16,  # ранг LoRA
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(model, lora_config)
    
    print("Trainable parameters:")
    model.print_trainable_parameters()
    
    return model

# настраиваем LoRA
model = setup_lora(model)

Trainable parameters:
trainable params: 16,777,216 || all params: 6,755,192,832 || trainable%: 0.2484


In [11]:
print("Подготовка обучающих данных...")
train_dataset = prepare_dataset(train_data, tokenizer, max_length=512)
val_dataset = prepare_dataset(val_data, tokenizer, max_length=512)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Покажем пример подготовленных данных
if len(train_dataset) > 0:
    print("\nПример подготовленного образца:")
    for key in train_dataset[0]:
        print(f"  {key}: {train_dataset[0][key][:50]}...")

Подготовка обучающих данных...
Train samples: 964
Val samples: 200

Пример подготовленного образца:
  input_ids: [1, 1, 518, 25580, 29962, 3532, 14816, 29903, 6778, 13, 3492, 526, 385, 17924, 297, 405, 2795, 14945, 3599, 3811, 654, 297, 1436, 749, 29889, 3529, 12439, 278, 5196, 29892, 9205, 2133, 29892, 322, 17015, 14945, 515, 278, 2183, 1426, 29889, 13, 29966, 829, 14816, 29903, 6778, 13, 13, 13438]...
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]...
  labels: [1, 1, 518, 25580, 29962, 3532, 14816, 29903, 6778, 13, 3492, 526, 385, 17924, 297, 405, 2795, 14945, 3599, 3811, 654, 297, 1436, 749, 29889, 3529, 12439, 278, 5196, 29892, 9205, 2133, 29892, 322, 17015, 14945, 515, 278, 2183, 1426, 29889, 13, 29966, 829, 14816, 29903, 6778, 13, 13, 13438]...


#### Обучение модели

In [12]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

In [13]:
# Аргументы обучения
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,                      # количество эпох
    per_device_train_batch_size=2,           # размер батча (уменьшите если не хватает памяти)
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,           # накопление градиентов
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=500,
    learning_rate=2e-4,
    fp16=True,                               # mixed precision
    report_to="none",                        # отключаем wandb
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print(f"Обучение на {len(train_dataset)} примерах")
print(f"Валидация на {len(val_dataset)} примерах")

Обучение на 964 примерах
Валидация на 200 примерах


In [14]:
try:
    trainer.train()
    print("Обучение завершено!")
except Exception as e:
    print(f"Ошибка при обучении: {e}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
100,0.669800,0.547516
200,0.516200,0.507634
300,0.388600,0.503463


Обучение завершено!


In [15]:
print("Сохраняем модель...")
model.save_pretrained(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")
print(f"Модель сохранена в {OUTPUT_DIR}/final_model")

Сохраняем модель...
Модель сохранена в ./llama2-ner-finetuned/final_model


#### Применение и оценка

In [16]:
def extract_entities_from_output(output_text):
    entities = {"PERSON": set(), "ORGANIZATION": set(), "LOCATION": set()}
    
    patterns = {
        "PERSON": r'\[Person\]:\s*(.+?)(?=\n\[|$)',
        "ORGANIZATION": r'\[Organization\]:\s*(.+?)(?=\n\[|$)',
        "LOCATION": r'\[Location\]:\s*(.+?)(?=\n\[|$)'
    }
    
    for entity_type, pattern in patterns.items():
        match = re.search(pattern, output_text, re.IGNORECASE | re.DOTALL)
        if match:
            entities_str = match.group(1).strip()
            if entities_str and entities_str != 'none':

                items = [item.strip() for item in entities_str.split(',') if item.strip()]
                entities[entity_type] = set(items)
    
    return entities

In [20]:
import re
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
import warnings


def evaluate_model(model, tokenizer, test_data, max_samples=None):
    model.eval()
    
    all_true_entities = []
    all_pred_entities = []
    
    device = next(model.parameters()).device
    print(f"Модель на устройстве: {device}")
    
    test_subset = test_data if max_samples is None else test_data[:max_samples]
    
    print(f"Оценка модели на {len(test_subset)} тестовых примерах...")
    
    for i, item in enumerate(tqdm(test_subset, desc="Обработка")):
        # формируем промпт без ответа
        prompt = format_llama2_chat(
            item["instruction"],
            item["input"]
        )
        
        inputs = tokenizer(prompt, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # декодируем ответ
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        if "[/INST]" in generated:
            response = generated.split("[/INST]")[-1].strip()
        else:
            response = generated
        
        pred_entities = extract_entities_from_output(response)
        true_entities = extract_entities_from_output(item["output"])
        
        for entity_type in ["PERSON", "ORGANIZATION", "LOCATION"]:
            all_true_entities.append(list(true_entities[entity_type]))
            all_pred_entities.append(list(pred_entities[entity_type]))
        
        if i < 2:
            print(f"\nПример {i+1}:")
            print(f"Текст: {item['input'][:80]}...")
            print(f"Истинные: {true_entities}")
            print(f"Предсказанные: {pred_entities}")
    
    print("Результаты оценки")
    
    all_labels = list(set([e for sublist in all_true_entities + all_pred_entities for e in sublist if e]))
    
    if all_labels:
        def to_binary(lists):
            return np.array([[1 if label in lst else 0 for label in all_labels] for lst in lists])
        
        y_true_bin = to_binary(all_true_entities)
        y_pred_bin = to_binary(all_pred_entities)
        
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true_bin, y_pred_bin, average="micro", zero_division=0
        )
        
        print(f"Micro Average:")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        
        # Метрики по типам
        print(f"\nПо типам сущностей:")
        for idx, etype in enumerate(["PERSON", "ORGANIZATION", "LOCATION"]):
            true_type = [all_true_entities[i] for i in range(idx, len(all_true_entities), 3)]
            pred_type = [all_pred_entities[i] for i in range(idx, len(all_pred_entities), 3)]
            
            type_labels = list(set([e for lst in true_type + pred_type for e in lst if e]))
            
            if type_labels:
                y_true = np.array([[1 if l in lst else 0 for l in type_labels] for lst in true_type])
                y_pred = np.array([[1 if l in lst else 0 for l in type_labels] for lst in pred_type])
                p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
                print(f"  {etype:12} -> P: {p:.3f} | R: {r:.3f} | F1: {f:.3f}")
    else:
        print("Нет сущностей для оценки")
    
    return precision, recall, f1

# Запускаем оценку
print("Начинаем оценку модели...")
precision, recall, f1 = evaluate_model(model, tokenizer, test_data, max_samples=len(test_data))

Начинаем оценку модели...
Модель на устройстве: cuda:0
Оценка модели на 303 тестовых примерах...


Обработка:   0%|          | 1/303 [00:03<17:25,  3.46s/it]


Пример 1:
Текст: Subordinated Loan Agreement - Silicium de Provence SAS and Evergreen Solar Inc ....
Истинные: {'PERSON': {'HERBERT SMITH'}, 'ORGANIZATION': {'EVERGREEN SOLAR', 'Silicium de Provence SAS', 'Evergreen Solar Inc', 'SILICIUM DE PROVENCE SAS'}, 'LOCATION': set()}
Предсказанные: {'PERSON': set(), 'ORGANIZATION': {'EVERGREEN SOLAR', 'INC', 'SILICIUM DE PROVENCE SAS'}, 'LOCATION': set()}


Обработка:   1%|          | 2/303 [00:05<13:17,  2.65s/it]


Пример 2:
Текст: SUBORDINATED LOAN AGREEMENT HERBERT SMITH LLP Page 1 of 12 7 - December 2007 TAB...
Истинные: {'PERSON': {'HERBERT SMITH'}, 'ORGANIZATION': set(), 'LOCATION': set()}
Предсказанные: {'PERSON': set(), 'ORGANIZATION': {'HERBERT SMITH LLP'}, 'LOCATION': set()}


Обработка: 100%|██████████| 303/303 [18:41<00:00,  3.70s/it] 


Результаты оценки
Micro Average:
Precision: 0.4651
Recall:    0.5769
F1-Score:  0.5150

По типам сущностей:
  PERSON       -> P: 0.803 | R: 0.790 | F1: 0.797
  ORGANIZATION -> P: 0.333 | R: 0.320 | F1: 0.327
  LOCATION     -> P: 0.068 | R: 0.176 | F1: 0.098


In [22]:
def test_custom_example(model, tokenizer, text):
    device = next(model.parameters()).device
    
    instruction = "You are an expert in Named Entity Recognition in finance. Please identify the Person, Organization, and Location Entity from the given text."
    
    prompt = format_llama2_chat(instruction, text)
    inputs = tokenizer(prompt, return_tensors="pt")
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "[/INST]" in generated:
        response = generated.split("[/INST]")[-1].strip()
    else:
        response = generated
    
    print(f"\nВходной текст: {text}")
    print(f"Ответ модели:")
    for i in response.split("\n"):
        if ("[") and ("]")  in i:
            print(i)
    
    return

examples = [
    "Apple Inc. reported record profits of $100 billion.",
    "While SpaceX has taken an unusual approach to its offering, setting up access for retail investors through brokerage firms at a level atypical in new deals typically dominated by institutions, the NASA fund is another alternative for investors to gain access to Elon Musk's rocket company.",
    "Mutual fund manager and billionaire Ron Baron, a long-time Tesla and SpaceX investor, owns the rocket company through his First Principles fund (RONB)",
    "John Smith works at Google in New York",
    "The borrower Sarah Johnson holds deposits of $50000",
    "Microsoft CEO Satya Nadella visited London yesterday",
    "This LOAN AND SECURITY AGREEMENT dated January 27, 1999, between SILICON VALLEY BANK ( Bank ), a California - chartered bank with its principal place of business at 3003 Tasman Drive, Santa Clara, California 95054 with a loan production office located at 40 William St., Ste."
]

print("Тестирование модели на пользовательских примерах:\n")

for example in examples:
    test_custom_example(model, tokenizer, example)

Тестирование модели на пользовательских примерах:


Входной текст: Apple Inc. reported record profits of $100 billion.
Ответ модели:
[Person]: none
[Organization]: Apple Inc
[Location]: none

Входной текст: While SpaceX has taken an unusual approach to its offering, setting up access for retail investors through brokerage firms at a level atypical in new deals typically dominated by institutions, the NASA fund is another alternative for investors to gain access to Elon Musk's rocket company.
Ответ модели:
[Person]: none
[Organization]: brokerage firm
[Location]: none

Входной текст: Mutual fund manager and billionaire Ron Baron, a long-time Tesla and SpaceX investor, owns the rocket company through his First Principles fund (RONB)
Ответ модели:
[Person]: Ron Baron
[Organization]: First Principles fund, RONB
[Location]: none

Входной текст: John Smith works at Google in New York
Ответ модели:
[Person]: John Smith
[Organization]: Google
[Location]: New York 1

Входной текст: The borrower

In [38]:
# Сохраняем результаты оценки
results = {
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_data)
}

with open(f"{OUTPUT_DIR}/metrics.json", 'w') as f:
    json.dump(results, f, indent=2)

print("✅ Метрики сохранены в metrics.json")
print(f"Результаты: {results}")

✅ Метрики сохранены в metrics.json
Результаты: {'precision': 0.2857142857142857, 'recall': 0.47619047619047616, 'f1_score': 0.35714285714285715, 'train_samples': 964, 'val_samples': 200, 'test_samples': 303}


#### Загрузка и применение модели

In [3]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast, Trainer, BitsAndBytesConfig,TrainingArguments
from peft import PeftModel  # 0.5.0

In [5]:
base_model = "./Llama-2-7b-chat-hb"
peft_model = "./llama2-ner-finetuned/final_model"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token



bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

print("✓ Загружаю модель")
model = LlamaForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="cuda:0",
    trust_remote_code=True
)
print("✓ Загружаю PEFT модель")
model = PeftModel.from_pretrained(model, peft_model)

print("✓ Модель успешно загружена")

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


✓ Загружаю модель


Loading checkpoint shards: 100%|██████████| 2/2 [00:33<00:00, 16.55s/it]


✓ Загружаю PEFT модель


E:\alena\fin_proj\fin_proj\lib\site-packages\peft\config.py:162: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'corda_config', 'ensure_weight_tying', 'lora_ga_config', 'monteclora_config', 'peft_version', 'qalora_group_size', 'target_parameters', 'trainable_token_indices', 'use_bdlora', 'use_qalora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


✓ Модель успешно загружена


In [6]:
model.eval()  # переводим в режим оценки

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

In [8]:
def format_llama2_chat(instruction, input_text, output_text=None):
    system_prompt = f"<<SYS>>\n{instruction}\n<</SYS>>\n\n"
    
    if output_text:
        # для обучения: полный промпт с ответом
        return f"<s>[INST] {system_prompt}{input_text} [/INST] {output_text} </s>"
    else:
        # для инференса: только вопрос
        return f"<s>[INST] {system_prompt}{input_text} [/INST]"

In [31]:
def test_custom_example(model, tokenizer, text):
    device = next(model.parameters()).device
    
    instruction = "You are an expert in Named Entity Recognition in finance. Please identify the Person, Organization, and Location Entity from the given text."
    
    prompt = format_llama2_chat(instruction, text)
    inputs = tokenizer(prompt, return_tensors="pt")
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "[/INST]" in generated:
        response = generated.split("[/INST]")[-1].strip()
    else:
        response = generated
    
    print(f"\nВходной текст: {text}")
    print(f"Ответ модели:")
    for i in response.split("\n"):
        if ("[") and ("]")  in i:
            print(i)
    
    return

# Тестируем на примерах
examples = [
    "Apple Inc. reported record profits of $100 billion.",
    "While SpaceX has taken an unusual approach to its offering, setting up access for retail investors through brokerage firms at a level atypical in new deals typically dominated by institutions, the NASA fund is another alternative for investors to gain access to Elon Musk's rocket company.",
    "Mutual fund manager and billionaire Ron Baron, a long-time Tesla and SpaceX investor, owns the rocket company through his First Principles fund (RONB)",
    "John Smith works at Google in New York",
    "The borrower Sarah Johnson holds deposits of $50000",
    "Microsoft CEO Satya Nadella visited London yesterday",
    "This LOAN AND SECURITY AGREEMENT dated January 27, 1999, between SILICON VALLEY BANK ( Bank ), a California - chartered bank with its principal place of business at 3003 Tasman Drive, Santa Clara, California 95054 with a loan production office located at 40 William St., Ste."
]

print("Тестирование модели на пользовательских примерах:\n")

for example in examples:
    test_custom_example(model, tokenizer, example)

Тестирование модели на пользовательских примерах:


Входной текст: Apple Inc. reported record profits of $100 billion.
Ответ модели:
[Person]: none
[Organization]: Apple Inc
[Location]: none

Входной текст: While SpaceX has taken an unusual approach to its offering, setting up access for retail investors through brokerage firms at a level atypical in new deals typically dominated by institutions, the NASA fund is another alternative for investors to gain access to Elon Musk's rocket company.
Ответ модели:
[Person]: none
[Organization]: SpaceX, NASA
[Location]: none

Входной текст: Mutual fund manager and billionaire Ron Baron, a long-time Tesla and SpaceX investor, owns the rocket company through his First Principles fund (RONB)
Ответ модели:
[Person]: Ron Baron
[Organization]: First Principles fund, RONB
[Location]: none 

Входной текст: John Smith works at Google in New York
Ответ модели:
[Person]: John Smith
[Organization]: Google
[Location]: New York 1

Входной текст: The borrower 